In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.transforms as mtransforms
import matplotlib.pyplot as plt

from scipy.stats import norm

In [ ]:
# Supress copy warning.
pd.options.mode.chained_assignment = None

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Select Variants of Interest

In [ ]:
sex = 'combined'
region = 'non_PAR'
label = 'GP2_FIGS_EUR_AJ' # GP2_EUR_AJ or GP2_FIGS_EUR_AJ

meta = pd.read_csv(f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}/locuszoom/merged/{label}_{region}_{sex}_filtered.tsv.gz', sep = '\t', compression = 'gzip')

vars_list = ['rs16999087',
            'rs28830934',
            'rs41306249',
            'rs5909697',
            'rs8689',
            'rs73459702']
target_vars = pd.DataFrame(meta[meta.rsid.isin(vars_list)].MarkerName)
display(target_vars)

### Forest Plots

For each variant of interest, build a forest plot showing per-study estimates plus the fixed- and random-effect meta-analysis summaries (P-values come from the METAL output; meta CIs are computed from the reported standard errors).

In [ ]:
# Per-study files that went into the meta-analysis
prepared_dir = f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/prepared'

study_files = {
    'GP2 AJ':   f'{prepared_dir}/{label}_{region}_{sex}_input01_XWAS_AJ_{region}_{sex}.PHENO.glm.logistic.hybrid.metal.tsv.gz',
    'GP2 EUR':  f'{prepared_dir}/{label}_{region}_{sex}_input02_XWAS_EUR_{region}_{sex}.PHENO.glm.logistic.hybrid.metal.tsv.gz',
    'FIGS AJ':  f'{prepared_dir}/{label}_{region}_{sex}_input03_XWAS_AJ_{region}_{sex}.PHENO.glm.logistic.hybrid.metal.tsv.gz',
    'FIGS EUR': f'{prepared_dir}/{label}_{region}_{sex}_input04_XWAS_EUR_{region}_{sex}.PHENO.glm.logistic.hybrid.metal.tsv.gz'
}

print(f"{region}: {len(study_files)} studies")

In [ ]:
target_snps = set(target_vars['MarkerName'].astype(str).tolist())

NEEDED_COLS = ['MARKER', 'A1', 'A1_FREQ', 'OBS_CT', 'BETA', 'SE', 'P']

def load_study_subset(filepath, target_snps):
    """Read a per-study GWAS file and return rows for target SNPs only."""
    df = pd.read_csv(filepath, sep='\t', usecols=NEEDED_COLS,
                     dtype={'MARKER': str, 'A1': str}, compression = 'gzip')
    df = df[df['MARKER'].isin(target_snps)].copy()
    return df

study_data = {s: load_study_subset(fp, target_snps) for s, fp in study_files.items()}

print({s: len(d) for s, d in study_data.items()})

In [ ]:
study_meta = meta[meta['MarkerName'].isin(target_snps)].set_index('MarkerName')

print('Meta hits:', len(study_meta))

In [ ]:
# Map: chr -> {pos -> gene} (optional). Pre-populate to label plots with gene
# names if known; otherwise the title falls back to rsID / marker name.
GENE_LOOKUP = {}  # e.g. {'X': {105296281: 'SLC44A1'}}

# Candidate column names in the METAL meta file (first one present wins).
# The random-effect row (METAL's *ARE columns) is drawn only if those
# columns are present.
META_COLS = {
    'a1':     ['Allele1', 'A1', 'alt', 'ALT'],
    'a2':     ['Allele2', 'A2', 'ref', 'REF'],
    'beta':   ['Effect', 'BETA', 'beta'],
    'or':     ['OR'],
    'se':     ['StdErr', 'SE', 'stderr_beta'],
    'p':      ['Pvalue', 'P-value', 'P', 'pvalue', 'p_value'],
    'beta_r': ['EffectARE', 'BETA(R)'],
    'or_r':   ['OR(R)'],
    'se_r':   ['StdErrARE', 'StdErrARe'],
    'p_r':    ['PvalueARE', 'P(R)'],
    'i2':     ['HetISq', 'I'],
    'q':      ['HetPVal', 'Q'],
    'rsid':   ['rsid', 'RSID', 'rsID'],
    'chrom':  ['chrom', 'CHROM', '#CHROM', 'Chromosome', 'CHR'],
    'pos':    ['pos', 'POS', 'Position', 'BP'],
}


def _meta_val(m, key):
    for c in META_COLS[key]:
        if c in m.index and pd.notna(m[c]):
            return m[c]
    return np.nan


def _meta_beta(m, beta_key, or_key):
    """Meta beta, falling back to log(OR) when only OR is reported."""
    beta = _meta_val(m, beta_key)
    if pd.isna(beta):
        odds = _meta_val(m, or_key)
        beta = np.log(odds) if pd.notna(odds) else np.nan
    return float(beta)


def _study_beta(r):
    """Per-study beta; PLINK .glm.logistic.hybrid files only report OR, so
    use log(OR). Their SE column is already LOG(OR)_SE."""
    if 'BETA' in r.index and pd.notna(r['BETA']):
        return r['BETA']
    if 'OR' in r.index and pd.notna(r['OR']):
        return np.log(r['OR'])
    return np.nan


def _locus_info(snp, m=None):
    """(chrom, pos, rsid) for the title, from the meta row if available,
    otherwise parsed from a chrom:pos:ref:alt marker name."""
    chrom = pos = rsid = None
    if m is not None:
        c, p, r = _meta_val(m, 'chrom'), _meta_val(m, 'pos'), _meta_val(m, 'rsid')
        chrom = str(c) if pd.notna(c) else None
        pos = int(p) if pd.notna(p) else None
        rsid = str(r) if pd.notna(r) else None
    parts = snp.split(':')
    if len(parts) >= 2 and (chrom is None or pos is None):
        try:
            chrom, pos = parts[0], int(parts[1])
        except ValueError:
            pass
    return chrom, pos, rsid


def _title(snp, chrom, pos, rsid, arrow_str, gene_name):
    lines = []
    if gene_name:
        lines.append(gene_name)
    head = rsid if rsid else snp
    if chrom is not None and pos is not None:
        head += f"  ({chrom}:{pos})"
    lines.append(f"{head}  {arrow_str}")
    return '\n'.join(lines)


def _format_p(p):
    if pd.isna(p):
        return 'NA'
    if p < 1e-3:
        return f"{p:.2e}"
    return f"{p:.3f}"


def _format_af(af):
    return f"{af:.3f}" if pd.notna(af) else "NA"

def _column_layout(study_labels, plot_w=6.0):
    """Size the table columns from the labels they actually have to hold.

    Everything is laid out in inches first (study names, AF / N, beta CI and
    P columns, with a fixed gap between each), then converted to the
    axes-fraction anchors used with the blended transform. This keeps the
    study names from running into the AF / N column no matter how long the
    study names are; the figure simply gets wider.
    """
    char_w = 0.085          # approx width of one character at fontsize 10
    pad = 0.15              # outer margin
    gap = 0.40              # gap between columns

    labels = list(study_labels) + ['Study']
    w_study = char_w * max(len(s) for s in labels)
    w_afn = char_w * len('0.000 / 000000')
    w_beta = char_w * len('-0.000 (-0.000, 0.000)')
    w_p = char_w * len('0.00e-00')

    x_study_l = pad
    x_afn_r = x_study_l + w_study + gap + w_afn
    ax_l = x_afn_r + gap
    ax_r = ax_l + plot_w
    x_beta_r = ax_r + gap + w_beta
    x_p_r = x_beta_r + gap + w_p
    fig_w = x_p_r + pad

    def frac(x_in):
        return (x_in - ax_l) / plot_w

    return {
        'fig_w': fig_w,
        'left': ax_l / fig_w,
        'right': ax_r / fig_w,
        'X_STUDY': frac(x_study_l),
        'X_AFN': frac(x_afn_r),
        'X_BETACI': frac(x_beta_r),
        'X_P': frac(x_p_r),
        'X_SEPL': frac(x_study_l - 0.05),
        'X_SEPR': frac(x_p_r + 0.05),
    }


def forest_plot(snp, study_data, meta_df,
                beta_label='Beta (log OR)',
                gene_name=None,
                save_path=None,
                se_thresh=100,
                sub_data=None,
                sub_parent='GP2 EUR'):
    """Plot a forest plot for `snp` (a meta MarkerName) across all studies in
    `study_data`, with fixed- and random-effect summaries from `meta_df`
    (METAL output, indexed by MarkerName).

    Optional `sub_data` ({label: df}, same columns as `study_data`) is drawn
    as indented rows under `sub_parent` (e.g. a per-cohort breakdown of a
    pooled study). These rows are display-only: they are excluded from k and
    the x-axis scale, and whiskers running off the axis end in an arrow.

    Per-study betas (log OR) are aligned to the meta Allele1 (sign flipped if
    A1 differs); studies with |SE| > se_thresh are shown as gray diamonds and
    excluded from the x-axis scale.

    Meta CIs use the reported StdErr / StdErrARE; if one is missing the SE
    is derived from the reported p-value so the displayed CI and p-value agree.
    """
    if snp not in meta_df.index:
        print(f"  [skip] {snp}: no meta result")
        return None
    m = meta_df.loc[snp]
    if isinstance(m, pd.DataFrame):
        m = m.iloc[0]

    chrom, pos_int, rsid = _locus_info(snp, m)
    if gene_name is None and chrom is not None:
        gene_name = GENE_LOOKUP.get(chrom, {}).get(pos_int)

    # METAL writes alleles in lower case.
    meta_a1 = str(_meta_val(m, 'a1')).upper()
    meta_a2 = _meta_val(m, 'a2')
    arrow_str = (f"{str(meta_a2).upper()} → {meta_a1}" if pd.notna(meta_a2)
                 else f"effect={meta_a1}")

    # Build per-study rows aligned to the meta effect allele.
    def _aligned_rows(data, is_sub):
        out = []
        for study, df in data.items():
            sub = df[df['MARKER'] == snp]
            if sub.empty:
                continue
            r = sub.iloc[0]
            beta = _study_beta(r); se = r['SE']; af = r['A1_FREQ']
            n = r['OBS_CT']; pv = r['P']; a1 = str(r['A1']).upper()
            if pd.isna(beta) or pd.isna(se):
                continue
            if a1 != meta_a1:
                beta = -beta
                af = 1 - af if pd.notna(af) else af
            out.append({'study': f"   {study}" if is_sub else study,
                        'beta': float(beta), 'se': float(se),
                        'af': float(af) if pd.notna(af) else np.nan,
                        'n': int(n), 'p': float(pv), 'sub': is_sub})
        return out

    rows = _aligned_rows(study_data, False)
    if not rows:
        print(f"  [skip] {snp}: no per-study results")
        return None

    sdf = pd.DataFrame(rows).sort_values('study').reset_index(drop=True)

    # Insert the breakdown rows directly under their parent study.
    sub_rows = _aligned_rows(sub_data, True) if sub_data else []
    if sub_rows and sub_parent in set(sdf['study']):
        subs = pd.DataFrame(sub_rows).sort_values('study')
        i = sdf.index[sdf['study'] == sub_parent][0] + 1
        sdf = pd.concat([sdf.iloc[:i], subs, sdf.iloc[i:]]).reset_index(drop=True)
    sdf['extreme'] = sdf['se'].abs() > se_thresh

    valid = sdf[~sdf['extreme'] & ~sdf['sub']].copy()
    k = len(valid)
    if k == 0:
        print(f"  [skip] {snp}: no studies under SE threshold")
        return None

    # Heterogeneity statistics (for the info line) — taken straight from the
    # METAL output: `HetPVal` is the p-value of Cochran's Q test and
    # `HetISq` is the I² heterogeneity index (in percent).
    Q = float(_meta_val(m, 'q'))
    I2 = float(_meta_val(m, 'i2'))

    beta_fixed  = _meta_beta(m, 'beta', 'or')
    beta_random = _meta_beta(m, 'beta_r', 'or_r')
    p_fixed  = float(_meta_val(m, 'p'))
    p_random = float(_meta_val(m, 'p_r'))

    def _se_from_p(beta, p):
        if not np.isfinite(beta) or not np.isfinite(p) or p <= 0 or p >= 1:
            return float('nan')
        if abs(beta) < 1e-12:
            return float('nan')
        z = norm.ppf(1 - p / 2)
        if z <= 0:
            return float('nan')
        return abs(beta) / z

    se_fixed = float(_meta_val(m, 'se'))
    if not np.isfinite(se_fixed):
        se_fixed = _se_from_p(beta_fixed, p_fixed)
    se_random = float(_meta_val(m, 'se_r'))
    if not np.isfinite(se_random):
        se_random = _se_from_p(beta_random, p_random)

    def _ci(beta, se):
        if not np.isfinite(se):
            return (beta, beta)
        return (beta - 1.96 * se, beta + 1.96 * se)

    ci_fixed  = _ci(beta_fixed,  se_fixed)
    ci_random = _ci(beta_random, se_random)

    # Summary rows to draw below the studies: (label, beta, ci, p, color).
    meta_rows = [('Fixed effect', beta_fixed, ci_fixed, p_fixed, '#c0392b')]
    if np.isfinite(beta_random):
        meta_rows.append(('Random effect', beta_random, ci_random, p_random, '#27ae60'))

    # Plot scale based on non-extreme studies + meta CIs
    meta_lo = [r[2][0] for r in meta_rows]
    meta_hi = [r[2][1] for r in meta_rows]
    x_lo = min((valid['beta'] - 1.96 * valid['se']).min(), *meta_lo)
    x_hi = max((valid['beta'] + 1.96 * valid['se']).max(), *meta_hi)
    span = x_hi - x_lo if x_hi > x_lo else 1.0
    x_lo -= 0.10 * span
    x_hi += 0.10 * span

    # Layout: studies stack from y=n_studies down to y=1; meta block below 0.
    n_studies = len(sdf)
    fig_h = max(6.0, 0.42 * n_studies + 4.0)

    # Column anchors (axes-fraction in x; data in y), sized so the study
    # names never overlap the AF / N column.
    lay = _column_layout(list(sdf['study']) + [r[0] for r in meta_rows])
    fig, ax = plt.subplots(figsize=(lay['fig_w'], fig_h))
    fig.subplots_adjust(left=lay['left'], right=lay['right'],
                        top=0.88, bottom=0.12)

    blended = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)

    X_STUDY  = lay['X_STUDY']
    X_AFN    = lay['X_AFN']
    X_BETACI = lay['X_BETACI']
    X_P      = lay['X_P']
    X_SEPL   = lay['X_SEPL']
    X_SEPR   = lay['X_SEPR']

    y_studies = np.arange(n_studies, 0, -1, dtype=float)
    y_meta_info = 0.2
    y_meta = [-0.8 - j for j in range(len(meta_rows))]
    y_last = y_meta[-1]

    # Header
    y_header = n_studies + 1.2
    ax.text(X_STUDY,  y_header, 'Study',      fontweight='bold',
            transform=blended, va='center', ha='left')
    ax.text(X_AFN,    y_header, 'AF / N',     fontweight='bold',
            transform=blended, va='center', ha='right')
    ax.text(X_BETACI, y_header, 'β (95% CI)', fontweight='bold',
            transform=blended, va='center', ha='right')
    ax.text(X_P,      y_header, 'P',          fontweight='bold',
            transform=blended, va='center', ha='right')
    ax.plot([X_SEPL, X_SEPR], [y_header - 0.5, y_header - 0.5],
            transform=blended, color='gray', lw=0.6, clip_on=False)

    # Per-study rows
    w_all = 1.0 / (valid['se'] ** 2) if k > 0 else pd.Series([1.0])
    max_w = float(w_all.max()) if len(w_all) else 1.0
    cap = 0.15
    for i, row in sdf.iterrows():
        y = y_studies[i]
        ax.text(X_STUDY, y, row['study'], transform=blended,
                va='center', ha='left',
                color='#555' if row['sub'] else 'black')
        ax.text(X_AFN, y, f"{_format_af(row['af'])} / {row['n']}",
                transform=blended, va='center', ha='right', color='#555')

        if row['extreme']:
            x_edge = x_hi - 0.04 * span if row['beta'] > 0 else x_lo + 0.04 * span
            ax.scatter([x_edge], [y], marker='D', color='lightgray',
                       s=80, zorder=5, clip_on=False)
            ax.text(X_BETACI, y, f"{row['beta']:.2f} [extreme SE]",
                    transform=blended, va='center', ha='right', color='#888')
            ax.text(X_P, y, _format_p(row['p']),
                    transform=blended, va='center', ha='right', color='#888')
        elif row['sub']:
            beta = row['beta']; se = row['se']
            ci_lo, ci_hi = beta - 1.96 * se, beta + 1.96 * se
            sub_c = '#8fb3d9'
            lo, hi = max(ci_lo, x_lo), min(ci_hi, x_hi)
            ax.plot([lo, hi], [y, y], color=sub_c, lw=1.0, zorder=2)
            for edge, clipped, mk in ((lo, ci_lo < x_lo, '<'), (hi, ci_hi > x_hi, '>')):
                if clipped:
                    ax.scatter([edge], [y], marker=mk, s=25, color=sub_c, zorder=3)
                else:
                    ax.plot([edge, edge], [y - cap * 0.7, y + cap * 0.7], color=sub_c, lw=1.0)
            if x_lo <= beta <= x_hi:
                size = min(200, 15 + 120 * (1.0 / se ** 2) / max_w)
                ax.scatter([beta], [y], s=size, marker='s', color=sub_c, zorder=3)
            ax.text(X_BETACI, y,
                    f"{beta:.3f} ({ci_lo:.3f}, {ci_hi:.3f})",
                    transform=blended, va='center', ha='right', color='#555')
            ax.text(X_P, y, _format_p(row['p']),
                    transform=blended, va='center', ha='right', color='#555')
        else:
            beta = row['beta']; se = row['se']
            ci_lo, ci_hi = beta - 1.96 * se, beta + 1.96 * se
            ax.plot([ci_lo, ci_hi], [y, y], color='#1f5f9e', lw=1.2, zorder=2)
            ax.plot([ci_lo, ci_lo], [y - cap, y + cap], color='#1f5f9e', lw=1.2)
            ax.plot([ci_hi, ci_hi], [y - cap, y + cap], color='#1f5f9e', lw=1.2)
            size = 25 + 175 * (1.0 / se ** 2) / max_w
            ax.scatter([beta], [y], s=size, marker='s',
                       color='#1f5f9e', zorder=3)

            ax.text(X_BETACI, y,
                    f"{beta:.3f} ({ci_lo:.3f}, {ci_hi:.3f})",
                    transform=blended, va='center', ha='right')
            ax.text(X_P, y, _format_p(row['p']),
                    transform=blended, va='center', ha='right')

    # Separator above the meta block
    ax.plot([X_SEPL, X_SEPR], [0.6, 0.6], transform=blended,
            color='gray', lw=0.6, clip_on=False)

    # Meta-analysis info line
    ax.text(X_STUDY, y_meta_info,
            f"Meta-analysis (k={k}  I²={I2:.1f}%  Q={_format_p(Q)})",
            transform=blended, va='center',
            style='italic', color='#444', fontsize=9)

    # Fixed- (and random-) effect: square + whisker
    meta_marker_size = 140
    for (lab, b, ci, p, col), y in zip(meta_rows, y_meta):
        ax.plot([ci[0], ci[1]], [y, y], color=col, lw=1.4, zorder=4)
        ax.plot([ci[0], ci[0]], [y - cap, y + cap], color=col, lw=1.4)
        ax.plot([ci[1], ci[1]], [y - cap, y + cap], color=col, lw=1.4)
        ax.scatter([b], [y], s=meta_marker_size, marker='s',
                   color=col, zorder=5)
        ax.text(X_STUDY, y, lab, fontweight='bold',
                transform=blended, va='center', ha='left', color=col)
        ax.text(X_BETACI, y,
                f"{b:.3f} ({ci[0]:.3f}, {ci[1]:.3f})",
                transform=blended, va='center', ha='right', color=col)
        ax.text(X_P, y, _format_p(p),
                transform=blended, va='center', ha='right', color=col)

    # Reference line at zero
    ax.axvline(x=0, color='gray', linestyle='--', lw=0.7, zorder=1)

    # Title
    ax.set_title(_title(snp, chrom, pos_int, rsid, arrow_str, gene_name),
                 fontsize=13, pad=18)

    # Axis cosmetics
    ax.set_xlim(x_lo, x_hi)
    n_sub = int(sdf['sub'].sum())
    y_bottom = y_last - (1.2 if sdf['extreme'].any() else 0.6) - (0.6 if n_sub else 0)
    ax.set_ylim(y_bottom, n_studies + 2.0)
    ax.set_xlabel(beta_label)
    ax.set_yticks([])
    for spine in ('left', 'top', 'right'):
        ax.spines[spine].set_visible(False)

    n_extreme = int(sdf['extreme'].sum())
    if n_extreme > 0:
        ax.text(0.5, y_last - 1.0,
                f"* {n_extreme} {'study' if n_extreme == 1 else 'studies'} "
                f"with |SE| > {se_thresh} shown as ◇ but excluded from "
                "x-axis scale",
                transform=mtransforms.blended_transform_factory(ax.transAxes, ax.transData),
                va='center', ha='center',
                style='italic', color='#888', fontsize=8)

    if n_sub > 0:
        ax.text(0.5, y_last - (1.6 if n_extreme > 0 else 1.0),
                f"Indented rows: per-cohort breakdown of {sub_parent} "
                "(display only; not part of the meta-analysis)",
                transform=mtransforms.blended_transform_factory(ax.transAxes, ax.transData),
                va='center', ha='center',
                style='italic', color='#888', fontsize=8)

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    # plt.show()
    plt.close(fig)
    return {
        'snp': snp, 'rsid': rsid, 'k': k, 'Q': Q, 'I2': I2,
        'beta_fixed': beta_fixed, 'se_fixed': se_fixed, 'p_fixed': p_fixed,
        'OR_fixed': float(np.exp(beta_fixed)),
        'beta_random': beta_random, 'se_random': se_random, 'p_random': p_random,
    }

In [ ]:
out_root = f'{wd}/working/R{rel}_prep/XWAS/meta/forest_plots/{label}/{region}_{sex}'
os.makedirs(out_root, exist_ok=True)

# Generate one forest plot per target variant.
target_list = target_vars['MarkerName'].astype(str).tolist()

summary = []
for snp in target_list:
    fname = snp.replace(':', '_')
    save = f'{out_root}/{fname}.png'
    res = forest_plot(snp, study_data, study_meta,
                      beta_label='Beta (log OR)',
                      save_path=save)
    if res is not None:
        summary.append(res)

summary_df = pd.DataFrame(summary)
summary_df.to_csv(f'{out_root}/{label}_{region}_{sex}_meta_summary.csv', index=False)

print(f"{region} plots: {len(summary_df)}")
display(summary_df.head())

### If no meta results available

In [ ]:
def forest_plot_no_meta(snp, study_data,
                        beta_label='Beta (log OR)',
                        gene_name=None,
                        save_path=None,
                        se_thresh=100,
                        effect_allele=None):
    """Forest plot for `snp` across studies, with no meta-analysis summary.

    Effect allele is taken from `effect_allele` if given, else from the SNP ID
    (chrom:pos:ref:alt -> alt), else the A1 of the first study. Per-study
    betas (log OR) are sign-flipped if their A1 differs from the chosen effect
    allele. Studies with |SE| > se_thresh are shown as gray diamonds at the edge.
    """
    chrom, pos_int, rsid = _locus_info(snp)
    parts = snp.split(':')
    ref, alt = (parts[2].upper(), parts[3].upper()) if len(parts) == 4 else (None, None)
    if effect_allele is None:
        effect_allele = alt
    if effect_allele is None:
        for df in study_data.values():
            hit = df[df['MARKER'] == snp]
            if not hit.empty:
                effect_allele = str(hit.iloc[0]['A1'])
                break
    effect_allele = str(effect_allele).upper()
    arrow_str = (f"{ref} → {alt}" if effect_allele == alt
                 else f"{alt} → {ref}" if alt is not None and effect_allele == ref
                 else f"effect={effect_allele}")
    if gene_name is None and chrom is not None:
        gene_name = GENE_LOOKUP.get(chrom, {}).get(pos_int)

    rows = []
    for study, df in study_data.items():
        sub = df[df['MARKER'] == snp]
        if sub.empty:
            continue
        r = sub.iloc[0]
        beta = _study_beta(r); se = r['SE']; af = r['A1_FREQ']
        n = r['OBS_CT']; pv = r['P']; a1 = str(r['A1']).upper()
        if pd.isna(beta) or pd.isna(se):
            continue
        if a1 != effect_allele:
            beta = -beta
            af = 1 - af if pd.notna(af) else af
        rows.append({'study': study,
                     'beta': float(beta), 'se': float(se),
                     'af': float(af) if pd.notna(af) else np.nan,
                     'n': int(n), 'p': float(pv)})
    if not rows:
        print(f"  [skip] {snp}: no per-study results")
        return None

    sdf = pd.DataFrame(rows).sort_values('study').reset_index(drop=True)
    sdf['extreme'] = sdf['se'].abs() > se_thresh
    valid = sdf[~sdf['extreme']].copy()
    if len(valid) == 0:
        print(f"  [skip] {snp}: no studies under SE threshold")
        return None

    x_lo = (valid['beta'] - 1.96 * valid['se']).min()
    x_hi = (valid['beta'] + 1.96 * valid['se']).max()
    span = x_hi - x_lo if x_hi > x_lo else 1.0
    x_lo -= 0.10 * span
    x_hi += 0.10 * span

    n_studies = len(sdf)
    fig_h = max(5.0, 0.42 * n_studies + 2.5)

    # Column anchors (axes-fraction in x; data in y), sized so the study
    # names never overlap the AF / N column.
    lay = _column_layout(list(sdf['study']))
    fig, ax = plt.subplots(figsize=(lay['fig_w'], fig_h))
    fig.subplots_adjust(left=lay['left'], right=lay['right'],
                        top=0.88, bottom=0.12)

    blended = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)

    X_STUDY  = lay['X_STUDY']
    X_AFN    = lay['X_AFN']
    X_BETACI = lay['X_BETACI']
    X_P      = lay['X_P']
    X_SEPL   = lay['X_SEPL']
    X_SEPR   = lay['X_SEPR']

    y_studies = np.arange(n_studies, 0, -1, dtype=float)

    y_header = n_studies + 1.2
    ax.text(X_STUDY,  y_header, 'Study',      fontweight='bold',
            transform=blended, va='center', ha='left')
    ax.text(X_AFN,    y_header, 'AF / N',     fontweight='bold',
            transform=blended, va='center', ha='right')
    ax.text(X_BETACI, y_header, 'β (95% CI)', fontweight='bold',
            transform=blended, va='center', ha='right')
    ax.text(X_P,      y_header, 'P',          fontweight='bold',
            transform=blended, va='center', ha='right')
    ax.plot([X_SEPL, X_SEPR], [y_header - 0.5, y_header - 0.5],
            transform=blended, color='gray', lw=0.6, clip_on=False)

    w = 1.0 / (valid['se'] ** 2)
    max_w = float(w.max()) if len(w) else 1.0
    for i, row in sdf.iterrows():
        y = y_studies[i]
        ax.text(X_STUDY, y, row['study'], transform=blended,
                va='center', ha='left')
        ax.text(X_AFN, y, f"{_format_af(row['af'])} / {row['n']}",
                transform=blended, va='center', ha='right', color='#555')

        if row['extreme']:
            x_edge = x_hi - 0.04 * span if row['beta'] > 0 else x_lo + 0.04 * span
            ax.scatter([x_edge], [y], marker='D', color='lightgray',
                       s=80, zorder=5, clip_on=False)
            ax.text(X_BETACI, y, f"{row['beta']:.2f} [extreme SE]",
                    transform=blended, va='center', ha='right', color='#888')
            ax.text(X_P, y, _format_p(row['p']),
                    transform=blended, va='center', ha='right', color='#888')
        else:
            beta = row['beta']; se = row['se']
            ci_lo, ci_hi = beta - 1.96 * se, beta + 1.96 * se
            ax.plot([ci_lo, ci_hi], [y, y], color='#1f5f9e', lw=1.2, zorder=2)
            cap = 0.15
            ax.plot([ci_lo, ci_lo], [y - cap, y + cap], color='#1f5f9e', lw=1.2)
            ax.plot([ci_hi, ci_hi], [y - cap, y + cap], color='#1f5f9e', lw=1.2)
            size = 25 + 175 * (1.0 / se ** 2) / max_w
            ax.scatter([beta], [y], s=size, marker='s',
                       color='#1f5f9e', zorder=3)
            ax.text(X_BETACI, y,
                    f"{beta:.3f} ({ci_lo:.3f}, {ci_hi:.3f})",
                    transform=blended, va='center', ha='right')
            ax.text(X_P, y, _format_p(row['p']),
                    transform=blended, va='center', ha='right')

    ax.axvline(x=0, color='gray', linestyle='--', lw=0.7, zorder=1)

    ax.set_title(_title(snp, chrom, pos_int, rsid, arrow_str, gene_name),
                 fontsize=13, pad=18)

    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(0.3, n_studies + 2.0)
    ax.set_xlabel(beta_label)
    ax.set_yticks([])
    for spine in ('left', 'top', 'right'):
        ax.spines[spine].set_visible(False)

    n_extreme = int(sdf['extreme'].sum())
    if n_extreme > 0:
        ax.text(0.5, 0.5,
                f"* {n_extreme} {'study' if n_extreme == 1 else 'studies'} "
                f"with |SE| > {se_thresh} shown as ◇ but excluded from x-axis scale",
                transform=mtransforms.blended_transform_factory(ax.transAxes, ax.transData),
                va='center', ha='center',
                style='italic', color='#888', fontsize=8)

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    # plt.show()
    plt.close(fig)
    return sdf

In [ ]:
def load_one_snp(filepath, locus_prefix):
    """Pull rows whose MARKER starts with `locus_prefix` (e.g. 'X:12345678:')."""
    df = pd.read_csv(filepath, sep='\t', usecols=NEEDED_COLS,
                     dtype={'MARKER': str, 'A1': str}, compression='gzip')
    return df[df['MARKER'].str.startswith(locus_prefix)].copy()


# Full MarkerName, or shorten to 'chrom:pos:' to catch every allele pair.
locus = target_vars['MarkerName'].iloc[0]

one_snp_data = {s: load_one_snp(fp, locus) for s, fp in study_files.items()}

found = set()
for d in one_snp_data.values():
    found.update(d['MARKER'].unique().tolist())
print('SNP IDs found:', sorted(found))
print(f'{region} rows:', {s: len(d) for s, d in one_snp_data.items()})

In [ ]:
# Pick the full SNP ID found above; if multiple, pass the one you want.
snp_id = sorted(found)[0]

out_dir = f'{out_root}/no_meta'
os.makedirs(out_dir, exist_ok=True)

forest_plot_no_meta(
    snp_id, one_snp_data,
    beta_label='Beta (log OR)',
    save_path=f"{out_dir}/{snp_id.replace(':', '_')}.png",
)

### Read in summary dataframes

In [ ]:
summary_df = pd.read_csv(f'{out_root}/{label}_{region}_{sex}_meta_summary.csv')
display(summary_df)

In [ ]:
# Per-study rows with implausibly large standard errors, shown as gray diamonds in the plots
se_thresh = 100

extreme_rows = []
for study, df in study_data.items():
    sub = df[df['SE'].abs() > se_thresh]
    for _, r in sub.iterrows():
        extreme_rows.append({
            'SNP': r['MARKER'], 'study': study,
            'A1': r['A1'], 'BETA': _study_beta(r), 'SE': r['SE'],
            'A1_FREQ': r['A1_FREQ'], 'OBS_CT': r['OBS_CT'], 'P': r['P'],
        })

extreme_df = pd.DataFrame(extreme_rows)
extreme_df.to_csv(f'{out_root}/{label}_{region}_{sex}_extreme_studies.csv', index=False)

print(f"Extreme (|SE|>{se_thresh}) per-study rows: {len(extreme_df)}")
print(f"Distinct SNPs affected: {extreme_df['SNP'].nunique() if len(extreme_df) else 0}")
if len(extreme_df):
    print('\nTop SNPs by # extreme studies:')
    display(extreme_df.groupby('SNP').size().sort_values(ascending=False).head(20))
    display(extreme_df.head(20))